In [0]:
# ----------------------------------------------------------------------------------------------------------------------------
# SCRIPT: 3.1a_alinhamento_frente_partido
# LOCAL: 3_gold/src/3_correlacoes_frentes_votos/
# OBJETIVO: Retorna se o deputado votou ou não diferente da orientação do partido. 
#           Fonte das tabelas geradas na 1_bronze/..3.1a_votacoes_ingest
# ENTREGA: Verificar se deputados de uma mesma frente votam de forma mais alinhada do que seus colegas de partido.
# ----------------------------------------------------------------------------------------------------------------------------

from pyspark.sql.functions import col, when

# Carregar os dados das tabelas
df_votos = spark.table("bronze_votos_deputados")
df_orient = spark.table("bronze_orientacoes_partido")
df_frentes = spark.table("gold_atlas_frentes_parlamentares")

# Renomear colunas
df_orient_clean = df_orient.select(
    col("siglaPartidoBloco").alias("partido_bloco"), 
    col("orientacaoVoto").alias("voto_partido")
)

# Cruzar Voto do Deputado com Orientação do Partido
df_analise = df_votos.select(
    col("deputado_.id").alias("id_deputado"),
    col("deputado_.siglaPartido").alias("partido_deputado"),
    col("tipoVoto").alias("voto_deputado")
).join(
    df_orient_clean, 
    col("partido_deputado") == col("partido_bloco"), 
    "left"
)

# Adicionar a informação das Frentes Parlamentares
df_final = df_analise.join(
    df_frentes.select("id_deputado", "nome_frente"), 
    "id_deputado", 
    "inner"
).withColumn(
    "seguiu_partido", 
    when(col("voto_deputado") == col("voto_partido"), 1).otherwise(0)
)

# Salvar na Gold
df_final.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_correlacao_frente_voto")

display(df_final)